In [ ]:
import sys
import warnings
from functools import partial

import numpy as np
import torch
import yaml
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor

sys.path.append("..")
from src.data_generation.data_preperation import data_preparation
from src.model.finetune import finetune
from src.evaluation.surface_eval import check_arbitrage_flat
cfg = yaml.safe_load(open("../config.yaml"))

warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
N_CONTEXT = 20
RUN_NAME = "ssvi_fixed_context_20"
data_provider = partial(data_preparation, cfg, n_context=N_CONTEXT)

In [ ]:
finetune(data_provider, run_name=RUN_NAME, n_epochs=100, n_surfaces_per_epoch=20, n_val_surfaces=10)

In [ ]:
N_TEST_SURFACES = 50
N_ESTIMATORS = 1 # finetuned with 1
test_train, test_test = data_preparation(cfg, N_TEST_SURFACES, N_CONTEXT)

In [ ]:
baseline = TabPFNRegressor(
    n_estimators=N_ESTIMATORS, inference_config={"FINGERPRINT_FEATURE": False},
)

finetuned = TabPFNRegressor(
    fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS,
    inference_config={"FINGERPRINT_FEATURE": False},
)
finetuned._initialize_model_variables()
finetuned_state = torch.load(f"../checkpoints/{RUN_NAME}/best.pt", map_location="cpu")
finetuned.model_.load_state_dict(finetuned_state)

In [ ]:
def eval_surfaces(model, train_list, test_list, reload_state=None):
    rmses, maes, mapes = [], [], []
    cal_violations, butterfly_violations = [], []
    for (X_tr, y_tr), (X_te, y_te) in zip(train_list, test_list):
        model.fit(X_tr, y_tr)
        if reload_state is not None:
            model.model_.load_state_dict(reload_state)
        y_pred = model.predict(X_te)
        rmses.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        maes.append(np.mean(np.abs(y_te - y_pred)))
        mapes.append(np.mean(np.abs((y_te - y_pred) / y_te)) * 100)

        cal_violation, butterfly_violation = check_arbitrage_flat(cfg, y_pred)
        cal_violations.append(cal_violation)
        butterfly_violations.append(butterfly_violation)

    return (
        np.mean(rmses), np.mean(maes), np.mean(mapes),
        np.mean(cal_violations), np.mean(butterfly_violations),
    )

In [ ]:
print("Evaluating baseline")
b_rmse, b_mae, b_mape, b_cal, b_butterfly = eval_surfaces(baseline, test_train, test_test)
print("Evaluating finetuned")
f_rmse, f_mae, f_mape, f_cal, f_butterfly = eval_surfaces(finetuned, test_train, test_test, reload_state=finetuned_state)

print(f"\n{'':20s} {'Baseline':>12s} {'Finetuned':>12s} {'Delta':>10s}")
print("-" * 56)
for name, b, f in [
    ("RMSE", b_rmse, f_rmse), ("MAE", b_mae, f_mae), ("MAPE (%)", b_mape, f_mape),
    ("Calendar arb (%)", b_cal * 100, f_cal * 100), ("Butterfly arb (%)", b_butterfly * 100, f_butterfly * 100),
]:
    print(f"{name:20s} {b:12.4f} {f:12.4f} {(f-b)/b * 100:+10.2f}%" if b else f"{name:20s} {b:12.4f} {f:12.4f} {'n/a':>10s}")

In [ ]:
N_CONTEXT_VALUES = [5, 8, 10, 15, 20, 40]
N_TEST_SURFACES_SWEEP = 50

sweep_results = []
for n_ctx in N_CONTEXT_VALUES:
    tr, te = data_preparation(cfg, N_TEST_SURFACES_SWEEP, n_ctx)

    b = eval_surfaces(baseline, tr, te)
    f = eval_surfaces(finetuned, tr, te, reload_state=finetuned_state)

    sweep_results.append((n_ctx, b, f))
    print(f"N_CONTEXT={n_ctx:>4d} | baseline RMSE={b[0]:.4f} MAPE={b[2]:.2f}% cal_arb={b[3]*100:.1f}% butterfly_arb={b[4]*100:.1f}% "
          f"| finetuned RMSE={f[0]:.4f} MAPE={f[2]:.2f}% cal_arb={f[3]*100:.1f}% butterfly_arb={f[4]*100:.1f}%")

print(f"\n{'N_CONTEXT':>10s} {'Base RMSE':>10s} {'fine RMSE':>10s} {'RMSE Δ':>8s} "
      f"{'Base MAPE':>10s} {'FT MAPE':>9s} {'MAPE Δ':>8s} "
      f"{'B cal%':>7s} {'F cal%':>7s} {'B bfly%':>8s} {'F bfly%':>8s}")
print("-" * 100)
for n_ctx, b, f in sweep_results:
    rmse_delta = (f[0] - b[0]) / b[0] * 100
    mape_delta = (f[2] - b[2]) / b[2] * 100
    print(f"{n_ctx:>10d} {b[0]:>10.4f} {f[0]:>10.4f} {rmse_delta:>7.1f}% "
          f"{b[2]:>10.4f} {f[2]:>9.4f} {mape_delta:>7.1f}% "
          f"{b[3]*100:>6.1f}% {f[3]*100:>6.1f}% {b[4]*100:>7.1f}% {f[4]*100:>7.1f}%")